# Apprendimento non supervisionato

In questo laboratorio proveremo a ottenere informazioni utili usando tecniche di apprendimento non supervisionato.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Caricamento e ispezione dei dati

In [ ]:
path_csv = "https://raw.githubusercontent.com/LorenzoTonet/Intro_to_ML_26-27/refs/heads/main/data/Dry_Bean_Dataset.csv"
df = pd.read_csv(path_csv)
df.head()

In [ ]:
df.describe()

Facciamo un po' di pre-processing dei dati:
- separiamo le feature numeriche
- codifichiamo le etichette vere
- dato che le scale sono molto diverse, le standardizziamo

In [ ]:
# Separo le feature dalle etichette
X_numerical = df.drop('Class', axis=1)
labels = df['Class'].values
X_train = X_numerical.values

# Codifico le etichette come interi
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(labels)

## Analisi delle Componenti Principali (PCA)

**Obiettivo**: trovare l'insieme di direzioni ortogonali lungo le quali la varianza dei dati è massima.

Riassunto del metodo:

* Centro la matrice delle feature $X$ calcolando $X_{\text{centered}} = X - \mu_X$ (oppure, in questo caso, $X_{\text{standardized}} = \frac{X - \mu_X}{\sigma_X}$);

* Calcolo la matrice di covarianza $C$ dei dati come $C=\frac{1}{N-1} X^T X$;

* Calcolo autovalori e autovettori della matrice di covarianza $C$ (usando la funzione `eigh` del modulo `scipy.LA`) e li ordino in base all'ordine decrescente degli autovalori. Li dispongo come colonne di una matrice $A$.

**Promemoria**: gli autovalori danno informazioni sulla varianza dei dati lungo la direzione dell'autovettore corrispondente.


## Centratura vs standardizzazione nella PCA

La **centratura** (sottrarre la media) è **sempre necessaria**. La PCA diagonalizza la matrice di covarianza e, senza centratura, la prima componente viene attratta verso la media dei dati invece che verso la direzione di massima varianza.

$$
X_c = X - \mathbf{1}\bar{x}^\top
$$

La **standardizzazione** (dividere anche per la deviazione standard) è un passo extra **opzionale** che cambia il problema: la PCA viene allora eseguita sulla matrice di correlazione invece che su quella di covarianza.

$$
X_s = X_c \, \mathrm{diag}(\sigma_1, \dots, \sigma_d)^{-1}
$$


In [ ]:
# 1. Calcolo media e deviazione standard di ogni feature
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)

# 2. Conservo una copia dei dati grezzi: mi serve dopo per il confronto
X_raw = X_train.copy()

# 3. Standardizzo i dati
X_train = (X_train - X_mean) / X_std

In [ ]:
class MyPCA:
    """PCA tramite decomposizione spettrale della matrice di covarianza."""
    # TODO: implemento l'algoritmo PCA
    def __init__(self):
        self.mean_ = None
        self.scale_ = None
        self.components_ = None
        self.eigenvalues_ = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        """Stimo media, componenti principali e varianza spiegata."""
        self.mean_ = X.mean(axis=0)
        self.scale_ = None
        Xc = X - self.mean_

        # 1. Matrice di covarianza (non distorta)
        C = (Xc.T @ Xc) / (Xc.shape[0] - 1)
        # 2. np.linalg.eigh: matrice simmetrica, autovalori in ordine crescente
        eigval, eigvec = np.linalg.eigh(C)

        # 3. Ordino per autovalore decrescente
        order = np.argsort(eigval)[::-1]

        self.eigenvalues_ = eigval[order]
        self.components_ = eigvec[:, order]

        # 4. Varianza spiegata: quota di ogni autovalore sul totale
        self.explained_variance_ratio_ = self.eigenvalues_ / self.eigenvalues_.sum()
        return self

    def transform(self, X, k=None):
        """Proietto X sulle prime k componenti principali."""
        # 1. Centro i dati usando la media appresa nel fit
        Xc = X - self.mean_

        # 2. Calcolo la proiezione X @ componenti
        Z = Xc @ self.components_

        # 3. Seleziono le prime k colonne
        Z = Z if k is None else Z[:, :k]
        return Z

pca = MyPCA().fit(X_train)

In [ ]:
cumulative_var = np.cumsum(pca.explained_variance_ratio_)
threshold = 0.95

# 1. Primo numero di componenti che raggiunge la soglia
k = np.argmax(cumulative_var >= threshold) + 1

print(f"Componenti necessarie per il {threshold:.0%} di varianza: {k}")

In [ ]:
eigenvalues = pca.eigenvalues_
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

components = np.arange(1, len(eigenvalues) + 1)

# 1. Scree plot: autovalori in funzione dell'indice della componente
plt.figure(figsize=(8, 4))
plt.plot(components, eigenvalues, marker="o")
plt.xlabel("indice componente")
plt.ylabel("autovalore")
plt.title("Grafico scree")
plt.xticks(components)
plt.grid(True, alpha=0.3)
plt.show()

# 2. Varianza cumulata spiegata
plt.figure(figsize=(8, 4))
plt.plot(components, cumulative_var, marker="o")
plt.axhline(y=threshold, linestyle="--", color="red", label="95%")
plt.xlabel("indice componente")
plt.ylabel("Varianza spiegata")
plt.title("Varianza spiegata")
plt.ylim(0, 1.05)
plt.xticks(components)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# 1. Proietto i dati sulle prime 3 componenti
new_X = pca.transform(X_train, k=3)

# 2. Disegno le prime due componenti, colorate con le etichette vere
plt.scatter(new_X[:, 0], new_X[:, 1], c=y, cmap='viridis', alpha=0.7)

In [ ]:
pca_raw = MyPCA().fit(X_raw)       # solo centratura (PCA sulla covarianza)
pca_std = MyPCA().fit(X_train)     # centratura + standardizzazione (PCA sulla correlazione)

def pc1(pca):
    """Restituisco i loadings della PC1 con segno fissato per il confronto."""
    v = pca.components_[:, 0]
    # Il segno dell'autovettore è arbitrario: lo fisso per poter confrontare i due casi
    return v * np.sign(v[np.argmax(np.abs(v))])

# 1. Confronto i loadings della PC1 nei due casi
d = X_train.shape[1]
idx = np.arange(d)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(idx - 0.2, pc1(pca_raw), width=0.4, label=f"Solo centratura (EVR={pca_raw.explained_variance_ratio_[0]:.2f})")
ax.bar(idx + 0.2, pc1(pca_std), width=0.4, label=f"Standardizzata (EVR={pca_std.explained_variance_ratio_[0]:.2f})")
ax.set_xlabel("Feature originale")
ax.set_ylabel("Loading della PC1")
ax.set_title("Composizione della PC1: grezza vs standardizzata")
ax.legend()
plt.show()

---

In [ ]:
from sklearn.decomposition import PCA

# 1. Addestro la PCA di scikit-learn (oggetto separato: NON sovrascrivo `pca`)
pca_sklearn = PCA()
X_pca = pca_sklearn.fit_transform(X_train)

# 2. Proiezione sulle prime due componenti
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    cmap="viridis",
    alpha=0.7
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Fagioli secchi: PCA")
plt.colorbar(scatter, label="Classe")
plt.show()

---

## Il primo algoritmo di clustering: K-means

Calcoleremo il clustering k-means usando due tipi di inizializzazione:

  1. Inizializzazione casuale: i centroidi dei cluster vengono inizializzati scegliendo punti casuali dal dataset;

  2. k-means++: scelgo il primo centro di cluster a caso, poi scelgo i nuovi centri in modo che siano lontani dai centri già esistenti.

# K-Means(X, K)

**Input:** dataset $X = \{x_1, \dots, x_N\} \subset \mathbb{R}^d$, numero di cluster $K$

**Output:** centroidi $\{\mu_1, \dots, \mu_K\}$ e assegnazioni ai cluster $\{c_1, \dots, c_N\}$

1. **Inizializzo** $K$ centroidi $\mu_1, \dots, \mu_K$ in modo casuale (es. campionando $K$ punti distinti da $X$).

2. **Ripeto fino a convergenza** (le assegnazioni non cambiano più, oppure i centroidi si spostano meno di una tolleranza $\varepsilon$):

   a. **Passo di assegnazione:** assegno ogni punto al centroide più vicino.

      $$
      c_i = \underset{k \in \{1, \dots, K\}}{\arg\min} \; \|x_i - \mu_k\|_2^2, \qquad i = 1, \dots, N
      $$

   b. **Passo di aggiornamento:** ricalcolo ogni centroide come media dei punti assegnati al suo cluster.

      $$
      \mu_k = \frac{1}{N_k} \sum_{i=1}^{N} \mathbb{1}[c_i = k] \, x_i, \qquad N_k = \sum_{i=1}^{N} \mathbb{1}[c_i = k]
      $$

3. **Restituisco** le assegnazioni $\{c_i\}$ e i centroidi $\{\mu_k\}$.

In [ ]:
def my_kmeans(X, k, max_iters=100, tol=1e-4):
    """Algoritmo di clustering K-means."""
    # TODO
    n_samples, n_features = X.shape
    labels = np.zeros(n_samples, dtype=int)

    # 1. Inizializzo i centroidi in modo casuale
    centroids_idx = np.random.choice(n_samples, size=k, replace=False)
    centroids = X[centroids_idx]
    new_centroids = np.zeros((k, n_features))

    for _ in range(max_iters):
        # 2. Assegno i cluster in base al centroide più vicino
        # Creo una matrice delle distanze da ogni campione a ogni centroide -> N x k
        distances = np.full((n_samples, k), np.inf)

        # 3. Calcolo le distanze da ogni campione a ogni centroide
        for j in range(k):
            distances[:, j] = np.linalg.norm(X - centroids[j], axis=1)

        # 4. Assegno le etichette in base al centroide più vicino
        labels = np.argmin(distances, axis=1)

        # 5. Aggiorno i centroidi
        for i in range(k):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                new_centroids[i] = cluster_points.mean(axis=0)
            else:
                # Se un cluster non ha punti assegnati, reinizializzo il suo centroide a caso
                new_centroids[i] = X[np.random.choice(n_samples)]

        # 6. Controllo la convergenza: mi fermo se i centroidi si sono mossi meno di tol
        converged = np.all(np.linalg.norm(new_centroids - centroids, axis=1) < tol)

        centroids = new_centroids
        if converged:
            break

    return labels, centroids

In [ ]:
labels, centroids = my_kmeans(new_X, k=6)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# 1. Classi originali
ax[0].scatter(new_X[:, 0], new_X[:, 1], c=y, cmap='viridis', alpha=0.7)
ax[0].set_title("Classi originali")

# 2. Cluster prodotti da K-means, con i centroidi evidenziati
ax[1].scatter(new_X[:, 0], new_X[:, 1], c=labels, cmap='viridis', alpha=0.7)
ax[1].set_title("Cluster K-means")
ax[1].scatter(centroids[:, 0], centroids[:, 1], c='red', marker='o', s=20, label='Centroidi')

### K-means e le sue varianti

- **K-means++:** migliora l'inizializzazione dei centroidi scegliendo centri iniziali che sono più probabilmente distanti tra loro. Questo porta spesso a risultati di clustering migliori e a una convergenza più rapida.

- **K-medoids:** simile a K-means, ma ogni centro del cluster (il *medoide*) è un punto reale del dataset, scelto per minimizzare la somma delle distanze dagli altri punti del cluster. È più robusto agli outlier e può usare metriche di distanza diverse.

- **Mini-batch K-means:** aggiorna i centroidi usando piccoli sottoinsiemi casuali dei dati invece dell'intero dataset a ogni iterazione. È più veloce e più scalabile sui dataset grandi.

- **Bisecting K-means:** parte con tutti i punti in un unico cluster e divide ripetutamente un cluster in due usando K-means, finché non raggiunge il numero di cluster desiderato.

**Differenza chiave:** K-means minimizza la somma delle distanze euclidee al quadrato all'interno dei cluster, mentre le sue varianti modificano l'inizializzazione, la scelta del rappresentante del cluster o la strategia di ottimizzazione.

---

# Giochiamo con altri algoritmi

In [ ]:
# 1. Carico MNIST
from sklearn.datasets import fetch_openml

X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
y = y.astype(int)

# 2. Estraggo un sottoinsieme di 5000 immagini e normalizzo i pixel in [0, 1]
idx = np.random.choice(len(X), 5000, replace=False)
X_sub = X[idx].astype(np.float32) / 255.0
y_sub = y[idx]

In [ ]:
# 1. Mostro una griglia di 18 immagini casuali con la rispettiva etichetta
fig, axes = plt.subplots(3, 6, figsize=(10, 6))
indices = np.random.choice(len(X), size=18, replace=False)

for ax, i in zip(axes.ravel(), indices):
    image = X[i].reshape(28, 28)

    ax.imshow(image, cmap="gray")
    ax.set_title(f"Etichetta: {y[i]}")
    ax.axis("off")

plt.suptitle("Immagini dal dataset MNIST", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA
# 1. Applico la PCA per ridurre a 2 dimensioni il subset MNIST
pca_mnist = PCA(n_components=2)
X_pca = pca_mnist.fit_transform(X_sub)

# 2. Proietto i dati e li coloro per cifra
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=y_sub, cmap="tab10", s=8, alpha=0.6
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(
    "MNIST — PCA bidimensionale "
    f"({pca_mnist.explained_variance_ratio_.sum():.1%} di varianza spiegata)"
)
plt.colorbar(scatter, ticks=range(10), label="Cifra")
plt.show()

# Isomap (Mappatura Isometrica delle Feature)
### di Tenenbaum et al (2000)

Isomap è un metodo di riduzione della dimensionalità **non lineare**. L'idea: i dati giacciono spesso su una **varietà** (manifold) a bassa dimensione, curva in uno spazio ad alta dimensione, quindi le distanze in linea retta (euclidee) sono fuorvianti. Isomap le sostituisce con le **distanze geodetiche**, cioè distanze misurate lungo la varietà, e poi trova un embedding che le preserva.

## Idea

1. **Grafo di vicinato.** Collego ogni punto $x_i$ ai suoi $k$ vicini più prossimi (o a tutti i punti entro un raggio $\varepsilon$), pesando ogni arco con la distanza euclidea $\|x_i - x_j\|$.

2. **Distanze geodetiche.** Approssimo la distanza lungo la varietà con il **cammino minimo** nel grafo tra ogni coppia di punti (Dijkstra o Floyd-Warshall):

   $$
   D^{G}_{ij} = \text{lunghezza del cammino minimo da } x_i \text{ a } x_j
   $$

3. **Embedding tramite MDS classico.** Cerco $y_i \in \mathbb{R}^m$ le cui distanze euclidee coincidono con $D^G$. Applico il double centering alla matrice delle distanze al quadrato,

   $$
   B = -\frac{1}{2} H \, (D^{G})^{\circ 2} \, H, \qquad H = I - \frac{1}{N}\mathbf{1}\mathbf{1}^\top
   $$

   e prendo i primi $m$ autovettori $v_l$ e autovalori $\lambda_l$ di $B$:

   $$
   Y = \left[\sqrt{\lambda_1}\, v_1, \dots, \sqrt{\lambda_m}\, v_m\right]
   $$

   Questo minimizza $\sum_{i,j} \left(D^G_{ij} - \|y_i - y_j\|\right)^2$ nel senso del MDS classico.

## Confronto tra Isomap e PCA

Isomap è essenzialmente **PCA/MDS applicato alle distanze geodetiche** invece che a quelle euclidee. Se i dati giacciono su un sottospazio piatto (lineare), distanze geodetiche ed euclidee coincidono e Isomap si riduce alla PCA.

## Avvertenze

- Sensibile alla scelta di $k$ (o di $\varepsilon$): un valore troppo grande crea archi di "cortocircuito" attraverso la varietà, uno troppo piccolo disconnette il grafo.
- Sensibile al rumore, e assume che la varietà sia **convessa** e isometrica a una regione piatta di $\mathbb{R}^m$.
- A differenza di t-SNE, preserva la struttura **globale**, ma è deterministico e la decomposizione spettrale costa $O(N^3)$.

In [ ]:
from sklearn.manifold import Isomap

# 1. Applico Isomap per ridurre a 2 dimensioni i dati
isomap = Isomap(n_components=2, n_neighbors=10)

X_isomap = isomap.fit_transform(X_sub)

# 2. Proietto i dati e li coloro per cifra
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    X_isomap[:, 0],
    X_isomap[:, 1],
    c=y_sub,
    cmap="tab10",
    s=8,
    alpha=0.6
)

plt.xlabel("Isomap 1")
plt.ylabel("Isomap 2")
plt.title("MNIST — Isomap bidimensionale")
plt.colorbar(scatter, ticks=range(10), label="Cifra")
plt.show()

# SNE (Embedding Stocastico dei Vicini)
### di Hinton e Roweis (2002)

SNE è un metodo di riduzione della dimensionalità **non lineare**. L'idea: trasformare le distanze in **probabilità di vicinato** sia nello spazio originale sia in quello di embedding, poi muovere i punti dell'embedding finché i due insiemi di probabilità non coincidono.

## Idea

1. **Affinità nello spazio ad alta dimensione.** La probabilità che $x_i$ scelga $x_j$ come vicino è una softmax pesata gaussianamente sulle distanze:

   $$
   p_{j|i} = \frac{\exp\left(-\|x_i - x_j\|^2 / 2\sigma_i^2\right)}{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / 2\sigma_i^2\right)}
   $$

2. **Affinità nello spazio a bassa dimensione.** La stessa costruzione per i punti dell'embedding $y_i \in \mathbb{R}^2$, con una banda fissata:

   $$
   q_{j|i} = \frac{\exp\left(-\|y_i - y_j\|^2\right)}{\sum_{k \neq i} \exp\left(-\|y_i - y_k\|^2\right)}
   $$

3. **Ottimizzazione.** Minimizzo la somma delle divergenze KL tra le due distribuzioni condizionate, usando la discesa del gradiente sui punti $y_i$:

   $$
   C = \sum_i \mathrm{KL}(P_i \,\|\, Q_i) = \sum_i \sum_j p_{j|i} \log \frac{p_{j|i}}{q_{j|i}}
   $$

   con gradiente

   $$
   \frac{\partial C}{\partial y_i} = 2 \sum_j \left(p_{j|i} - q_{j|i} + p_{i|j} - q_{i|j}\right)(y_i - y_j)
   $$

## Da SNE a t-SNE

t-SNE mantiene lo stesso schema ma cambia le distribuzioni ottimizzate: simmetrizza le affinità ad alta dimensione in una distribuzione congiunta $p_{ij}$ e sostituisce la gaussiana nello spazio a bassa dimensione con una distribuzione **Student-t** a code pesanti, $q_{ij} \propto (1 + \|y_i - y_j\|^2)^{-1}$. Minimizza poi $\mathrm{KL}(P \,\|\, Q)$ su queste distribuzioni congiunte, il che dà un gradiente più semplice e attenua il problema del crowding.

In [ ]:
from sklearn.manifold import TSNE

# 1. Applico t-SNE ai dati MNIST
X_tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    max_iter=1000
).fit_transform(X_sub)

# 2. Proietto i dati e li coloro per cifra
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=y_sub, cmap="tab10", s=8, alpha=0.6
)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("MNIST — t-SNE bidimensionale")
plt.colorbar(scatter, ticks=range(10), label="Cifra")
plt.show()

---

## Altre risorse
* [video che spiega PCA vs t-SNE vs UMAP](https://youtu.be/o_cAOa5fMhE?si=9vvkppYAbKtFpLRn)
*   [Effetto del riscalamento nella PCA](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)
*   [Matematica della PCA](https://youtu.be/dhK8nbtii6I?si=8PCuyZn0kT-BLuW3)
*   [t-SNE](https://youtu.be/NEaUSP4YerM?si=HZx3uzCCxG8oNQzC)
*   [Autovettori e autovalori](https://youtu.be/PFDu9oVAE-g?si=Ubv-fdszu1_Lqyjp)